# demand forecast EDA

Quick look at the daily POS rollup used as the source for our KFP pipeline. Focus areas:

- coverage of stores and SKUs
- seasonality (weekly + yearly) that motivates prophet as a baseline
- lag structure that motivates the xgboost feature set
- outliers and stock-out zeros (bias downward if we do not mask them)

In [ ]:
import pandas as pd
from google.cloud import bigquery
client = bigquery.Client()
df = client.query('''
SELECT store_id, sku_id, DATE(sale_date) AS ds, SUM(units_sold) AS units
FROM `retail_raw.pos_daily`
WHERE sale_date BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 180 DAY) AND CURRENT_DATE()
GROUP BY 1,2,3
''').to_dataframe()
df.shape

In [ ]:
import matplotlib.pyplot as plt
agg = df.groupby('ds').units.sum().reset_index()
agg.plot(x='ds', y='units', figsize=(11,3), title='total units / day')
plt.show()

clear weekly seasonality with weekend peaks. yearly seasonality visible around holidays
in Nov/Dec. this is what prophet catches out of the box; xgboost with lag + rolling window
features should improve on it for higher-cadence SKUs.